# Notebook 13 — Hands-On Lab & Project Milestone 2

**Difficulty:** ⭐⭐⭐⭐⭐ | **Estimated Time:** 2.5 hours

---

## Lab Overview

This notebook combines two deliverables:

**Part A — Hands-On Lab: Feature Engineering on Mixed-Type Data**  
You will engineer features from 12 raw Airbnb columns and demonstrate a ≥ 5% R² improvement over the raw-feature baseline using Ridge regression alone.

**Part B — Project Milestone 2: End-to-End ML Prediction System**  
You will compare 5 algorithms, tune the best one, build a production Pipeline, simulate a REST endpoint, write a Model Card, and implement data-drift monitoring.

---

> ### 🏁 Milestone 2 Badge
> Complete all deliverables in Part B to earn the **ML Engineer** milestone badge.
> Checklist in Cell 40.

**Week 12 Theme:** Airbnb-style listing price prediction


## Business Narrative

**PropInsight Analytics** manages pricing intelligence for 5,000 Airbnb-style listings across a major city.

Their current **rule-based system** prices listings as:
```
price = bedrooms × £50 + location_bonus
```
This achieves a test-set **RMSE of £42** — but it ignores review quality, host experience, property type, neighbourhood amenities, and dozens of other signals.

**Your mission:** Build a machine-learning pricing engine that beats **£30 RMSE** using:
1. Feature engineering on the 12 raw features
2. A 5-algorithm comparison
3. Hyperparameter tuning on the best algorithm
4. A production-ready serialised Pipeline
5. A REST-style prediction endpoint
6. A Model Card documenting intended use, limitations, and performance

The rule-based team will benchmark against your RMSE. Beat £30 and the ML system goes to production.


In [ ]:
# ── All imports and synthetic dataset generation ──
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import json
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge, Lasso
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import mutual_info_regression
from sklearn.model_selection import cross_val_score, KFold, train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.neighbors import BallTree

sns.set_theme(style='whitegrid')
np.random.seed(42)

# ── Synthetic Airbnb dataset: 5000 listings, 12 raw features ──
N = 5000
rng = np.random.default_rng(42)

# Numeric features
bedrooms             = rng.choice([1, 2, 3, 4, 5], N, p=[0.30, 0.35, 0.20, 0.10, 0.05])
bathrooms            = np.clip(bedrooms * 0.6 + rng.normal(0, 0.3, N), 0.5, 5.0).round(1)
accommodates         = np.clip(bedrooms * 1.8 + rng.integers(-1, 2, N), 1, 16).astype(int)
review_score         = np.clip(rng.normal(4.5, 0.4, N), 1.0, 5.0).round(2)
num_reviews          = rng.negative_binomial(5, 0.3, N).astype(float)
host_experience_days = np.clip(rng.exponential(800, N), 30, 3650).astype(int)

# Boolean
is_superhost = rng.choice([0, 1], N, p=[0.65, 0.35]).astype(int)

# Categorical
room_type = rng.choice(
    ['Entire home/apt', 'Private room', 'Shared room'], N,
    p=[0.55, 0.38, 0.07]
)
neighbourhood_group = rng.choice(
    ['Westminster', 'Camden', 'Tower Hamlets', 'Hackney',
     'Southwark', 'Lambeth', 'Islington', 'Wandsworth'],
    N,
    p=[0.18, 0.14, 0.13, 0.12, 0.12, 0.11, 0.11, 0.09]
)

# Geographic — centred on London
latitude  = rng.normal(51.51, 0.08, N)
longitude = rng.normal(-0.12, 0.10, N)

# Text description
desc_pool = [
    'cosy studio near transport links excellent wifi',
    'beautiful apartment with panoramic city views rooftop',
    'budget room shared bathroom close to centre',
    'modern flat fully equipped kitchen fast wifi',
    'luxury penthouse rooftop terrace chef kitchen skyline',
    'quiet neighbourhood parks family friendly garden',
    'professional workspace conference fast wifi ergonomic',
    'charming Victorian house private garden fireplace',
    'basic room affordable budget backpacker shared',
    'penthouse suite luxury concierge panoramic views chef',
]
description = rng.choice(desc_pool, N)

# ── Price generation with 4 realistic segments ──
location_bonus = np.where(neighbourhood_group == 'Westminster', 0.55,
                 np.where(neighbourhood_group == 'Camden', 0.35,
                 np.where(neighbourhood_group == 'Islington', 0.25, 0.0)))
room_bonus     = np.where(room_type == 'Entire home/apt',  0.50,
                 np.where(room_type == 'Private room',      0.10, -0.30))
superhost_bonus = is_superhost * 0.15
exp_bonus       = np.log1p(host_experience_days) * 0.05

log_price = (
    3.60
    + 0.28 * bedrooms
    + 0.12 * review_score
    + 0.025 * np.log1p(num_reviews)
    + location_bonus
    + room_bonus
    + superhost_bonus
    + exp_bonus
    + rng.normal(0, 0.22, N)
)

df_raw = pd.DataFrame({
    'bedrooms':             bedrooms.astype(float),
    'bathrooms':            bathrooms,
    'accommodates':         accommodates.astype(float),
    'review_score':         review_score,
    'num_reviews':          num_reviews,
    'host_experience_days': host_experience_days.astype(float),
    'is_superhost':         is_superhost.astype(float),
    'room_type':            room_type,
    'neighbourhood_group':  neighbourhood_group,
    'latitude':             latitude,
    'longitude':            longitude,
    'description':          description,
    'log_price':            log_price,
})

# Inject ~4% missing values in numeric columns
for col in ['bedrooms', 'bathrooms', 'review_score', 'num_reviews', 'host_experience_days']:
    mask = rng.random(N) < 0.04
    df_raw.loc[mask, col] = np.nan

print(f"Dataset shape : {df_raw.shape}")
print(f"\nColumn dtypes :\n{df_raw.dtypes}")
print(f"\nMissing rates (%) :\n{(df_raw.isnull().mean() * 100).round(1)}")

In [ ]:
np.random.seed(42)

# ── EDA: 4-panel overview ──
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# (a) log_price distribution
ax = axes[0, 0]
ax.hist(df_raw['log_price'], bins=50, color='steelblue', edgecolor='white', alpha=0.85)
ax.axvline(df_raw['log_price'].mean(), color='red', linestyle='--', label=f"Mean={df_raw['log_price'].mean():.2f}")
ax.set_xlabel('log(price)')
ax.set_ylabel('Count')
ax.set_title('(a) Log-Price Distribution')
ax.legend()

# (b) price by room_type — show in original £ scale
ax = axes[0, 1]
price_gbp = np.expm1(df_raw['log_price'])          # convert back to £
room_order = ['Entire home/apt', 'Private room', 'Shared room']
room_data  = [price_gbp[df_raw['room_type'] == r].values for r in room_order]
bp = ax.boxplot(room_data, labels=room_order, patch_artist=True,
                boxprops=dict(facecolor='lightsteelblue'))
ax.set_xlabel('Room Type')
ax.set_ylabel('Price (£/night)')
ax.set_title('(b) Price by Room Type')
ax.tick_params(axis='x', rotation=15)

# (c) review_score vs. log_price scatter (sample 500 for clarity)
ax = axes[1, 0]
sample_idx = np.random.choice(N, 500, replace=False)
ax.scatter(
    df_raw['review_score'].iloc[sample_idx],
    df_raw['log_price'].iloc[sample_idx],
    alpha=0.3, s=15, color='darkorange'
)
ax.set_xlabel('Review Score')
ax.set_ylabel('log(price)')
ax.set_title('(c) Review Score vs. log-Price (sample n=500)')

# (d) Correlation heatmap of numeric features
ax = axes[1, 1]
numeric_cols_eda = ['bedrooms', 'bathrooms', 'accommodates', 'review_score',
                    'num_reviews', 'host_experience_days', 'is_superhost', 'log_price']
corr_matrix = df_raw[numeric_cols_eda].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))  # upper triangle mask
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f',
    cmap='coolwarm', center=0, ax=ax, cbar_kws={'shrink': 0.8}
)
ax.set_title('(d) Correlation Heatmap')

plt.suptitle('Exploratory Data Analysis — PropInsight Listings', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
np.random.seed(42)

# ── Temporal-style train/test split: first 4000 = train, last 1000 = test ──
# In practice, newer listings are unseen at training time — we simulate this
# by treating row order as a proxy for listing creation time.
TRAIN_SIZE = 4000

df_train = df_raw.iloc[:TRAIN_SIZE].copy().reset_index(drop=True)
df_test  = df_raw.iloc[TRAIN_SIZE:].copy().reset_index(drop=True)

# Features and target
TARGET = 'log_price'
raw_feature_cols = [
    'bedrooms', 'bathrooms', 'accommodates', 'review_score', 'num_reviews',
    'host_experience_days', 'is_superhost', 'room_type', 'neighbourhood_group',
    'latitude', 'longitude', 'description'
]

X_train_raw = df_train[raw_feature_cols].copy()
y_train     = df_train[TARGET].copy()
X_test_raw  = df_test[raw_feature_cols].copy()
y_test      = df_test[TARGET].copy()

print(f"Train set : {X_train_raw.shape}  (rows 0–{TRAIN_SIZE-1})")
print(f"Test set  : {X_test_raw.shape}  (rows {TRAIN_SIZE}–{N-1})")
print()
print("Target (log_price) stats:")
print(f"  Train — mean={y_train.mean():.3f}, std={y_train.std():.3f}")
print(f"  Test  — mean={y_test.mean():.3f},  std={y_test.std():.3f}")

## Part A — Feature Engineering Lab

We will engineer features in five stages:

1. **Numeric preprocessing** — clip outliers, log-transform skewed columns
2. **Ratio features** — capture relationships between raw columns
3. **Geographic features** — distance to centre, listing density
4. **Interaction features** — multiplicative combinations
5. **Text features** — TF-IDF, length, and keyword flags
6. **Target encoding** — encode high-cardinality categorical with CV-based smoothing

After each stage we will measure the running R² improvement over the raw baseline.


In [ ]:
np.random.seed(42)

# ── Stage 1: Numeric preprocessing — clip outliers, log-transform skewed columns ──
# Work on a copy of the full dataset so we can track all feature stages
df = df_raw.copy()

skewed_cols = ['num_reviews', 'host_experience_days', 'accommodates']

# Compute 1st / 99th percentiles on training rows only (no leakage)
percentiles = {}
for col in ['bedrooms', 'bathrooms', 'accommodates', 'num_reviews', 'host_experience_days']:
    p01 = df_train[col].quantile(0.01)
    p99 = df_train[col].quantile(0.99)
    percentiles[col] = (p01, p99)
    # Apply clipping to full dataset using training percentiles
    df[col] = df[col].clip(lower=p01, upper=p99)

# Before/after log transform for two features
fig, axes = plt.subplots(2, 2, figsize=(12, 7))

for row_idx, col in enumerate(['num_reviews', 'host_experience_days']):
    original_vals = df_raw[col].dropna().values
    log_vals      = np.log1p(df[col].fillna(df[col].median()).values)

    axes[row_idx, 0].hist(original_vals, bins=50, color='salmon', edgecolor='white', alpha=0.8)
    axes[row_idx, 0].set_title(f'{col} — original (right-skewed)')
    axes[row_idx, 0].set_xlabel('Value')

    axes[row_idx, 1].hist(log_vals, bins=50, color='steelblue', edgecolor='white', alpha=0.8)
    axes[row_idx, 1].set_title(f'log1p({col}) — after transform')
    axes[row_idx, 1].set_xlabel('log1p(Value)')

plt.suptitle('Before / After Log Transform', fontweight='bold')
plt.tight_layout()
plt.show()

# Apply log transform to skewed columns (after clipping)
for col in skewed_cols:
    df[f'{col}_log'] = np.log1p(df[col].fillna(df[col].median()))

print("Numeric preprocessing complete.")
print(f"Skewness of num_reviews before: {df_raw['num_reviews'].skew():.2f}")
print(f"Skewness of num_reviews after : {df['num_reviews_log'].skew():.2f}")

In [ ]:
np.random.seed(42)

# ── Stage 2: Ratio features — capture structural relationships ──
# These features express insights a domain expert would compute manually.

df['accommodates_per_bedroom'] = (
    df['accommodates'] / df['bedrooms'].clip(lower=1)          # guests per bedroom
)
df['bathrooms_per_bedroom'] = (
    df['bathrooms'] / df['bedrooms'].clip(lower=1)             # luxury indicator
)
df['review_credibility'] = (
    df['review_score'] * np.log1p(df['num_reviews'])           # high score + many reviews
)
df['occupancy_proxy'] = (
    df['num_reviews']
    / (df['host_experience_days'].fillna(df['host_experience_days'].median()) / 30 + 0.1)
)  # reviews per month of hosting — proxy for actual occupancy rate

df['host_tenure_years'] = (
    df['host_experience_days'].fillna(df['host_experience_days'].median()) / 365
)

ratio_features = ['accommodates_per_bedroom', 'bathrooms_per_bedroom',
                  'review_credibility', 'occupancy_proxy', 'host_tenure_years']

print("Ratio features created:")
print(df[ratio_features].describe().round(3))

In [ ]:
np.random.seed(42)

# ── Stage 3: Geographic features ──

def haversine(lat1, lon1, lat2, lon2):
    """Compute great-circle distance in km between two lat/lon points."""
    R = 6371.0                                            # Earth radius in km
    phi1 = np.radians(lat1)
    phi2 = np.radians(lat2)
    dphi    = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)
    a = (np.sin(dphi / 2) ** 2
         + np.cos(phi1) * np.cos(phi2) * np.sin(dlambda / 2) ** 2)
    return 2 * R * np.arcsin(np.sqrt(a))


# Distance to London city centre (Charing Cross: 51.5080, -0.1247)
CENTRE_LAT, CENTRE_LON = 51.5080, -0.1247
df['distance_to_centre_km'] = haversine(
    df['latitude'].values, df['longitude'].values,
    CENTRE_LAT, CENTRE_LON
)

# ── Listing density within 1 km using BallTree (radians for haversine metric) ──
coords_rad = np.radians(df[['latitude', 'longitude']].values)   # BallTree expects radians
tree = BallTree(coords_rad, metric='haversine')

RADIUS_KM = 1.0
EARTH_R   = 6371.0
radius_rad = RADIUS_KM / EARTH_R                               # convert km to radians

counts = tree.query_radius(coords_rad, r=radius_rad, count_only=True)
df['listing_density_1km'] = counts.astype(float) - 1           # subtract self

print("Geographic features created:")
print(df[['distance_to_centre_km', 'listing_density_1km']].describe().round(3))

# Quick scatter: distance to centre vs. price
fig, ax = plt.subplots(figsize=(8, 4))
sample_idx = np.random.choice(N, 500, replace=False)
sc = ax.scatter(
    df['distance_to_centre_km'].iloc[sample_idx],
    df['log_price'].iloc[sample_idx],
    alpha=0.3, s=15, c=df['listing_density_1km'].iloc[sample_idx],
    cmap='YlOrRd'
)
plt.colorbar(sc, ax=ax, label='Listing density (1km radius)')
ax.set_xlabel('Distance to City Centre (km)')
ax.set_ylabel('log(price)')
ax.set_title('Geography vs. Price: Central Listings Command a Premium')
plt.tight_layout()
plt.show()

In [ ]:
np.random.seed(42)

# ── Stage 4: Interaction features ──
# Interactions capture joint effects that neither feature explains alone.
# E.g., a superhost with high reviews is worth more than either factor separately.

df['superhost_x_review'] = (
    df['is_superhost'] * df['review_score']                    # superhost quality signal
)
df['bedrooms_x_accommodates'] = (
    df['bedrooms'] * df['accommodates']                        # overall listing scale
)
df['credibility_x_distance'] = (
    df['review_credibility']
    / (df['distance_to_centre_km'] + 0.1)                     # high credibility near centre matters most
)

interaction_features = [
    'superhost_x_review', 'bedrooms_x_accommodates', 'credibility_x_distance'
]
print("Interaction features created:")
print(df[interaction_features].describe().round(3))

In [ ]:
np.random.seed(42)

# ── Stage 5: Text features from listing description ──

# Fill any null descriptions
df['description'] = df['description'].fillna('')

# (a) TF-IDF: fit only on training rows
tfidf = TfidfVectorizer(max_features=50, stop_words='english')
tfidf.fit(df.iloc[:TRAIN_SIZE]['description'])                 # fit on train only
tfidf_matrix = tfidf.transform(df['description'])             # transform all rows

# (b) Description length in characters
df['desc_length'] = df['description'].str.len()

# (c) Boolean luxury keyword flag
luxury_words = ['panoramic', 'rooftop', 'chef', 'penthouse', 'concierge', 'luxury', 'suite']
df['has_luxury_words'] = df['description'].str.lower().str.contains(
    '|'.join(luxury_words), regex=True
).astype(int)

# (d) Boolean budget keyword flag
budget_words = ['budget', 'basic', 'shared', 'affordable', 'backpacker']
df['has_budget_words'] = df['description'].str.lower().str.contains(
    '|'.join(budget_words), regex=True
).astype(int)

print(f"TF-IDF vocabulary size : 50 features")
print(f"Luxury keyword prevalence : {df['has_luxury_words'].mean():.2%}")
print(f"Budget keyword prevalence : {df['has_budget_words'].mean():.2%}")
print(f"Description length (mean): {df['desc_length'].mean():.0f} chars")

In [ ]:
np.random.seed(42)

# ── Stage 6: K-fold target encoding for neighbourhood_group ──
# Target encoding replaces a category with the mean target value for that category.
# K-fold variant: each training row is encoded using folds that exclude its own fold
# — this prevents the encoder from leaking target information about itself.
# Smoothing blends the category mean toward the global mean when n is small.

def kfold_target_encode(train_series, train_target, test_series, n_splits=5, smoothing=5):
    """K-fold target encoding with smoothing.

    Returns encoded train array, encoded test array, and per-category means dict.
    """
    global_mean   = train_target.mean()
    train_encoded = np.zeros(len(train_series))  # output for training rows
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

    for fold_train_idx, fold_val_idx in kf.split(train_series):
        # Compute category means using only the fold's training rows
        fold_train_s  = train_series.iloc[fold_train_idx]
        fold_train_t  = train_target.iloc[fold_train_idx]
        cat_stats     = fold_train_t.groupby(fold_train_s).agg(['mean', 'count'])

        # Smooth: blend category mean with global mean
        cat_stats['smoothed'] = (
            (cat_stats['mean'] * cat_stats['count'] + global_mean * smoothing)
            / (cat_stats['count'] + smoothing)
        )
        # Encode the validation rows using this fold's statistics
        fold_val_cats = train_series.iloc[fold_val_idx]
        train_encoded[fold_val_idx] = fold_val_cats.map(
            cat_stats['smoothed']
        ).fillna(global_mean).values

    # For test encoding: use means from the full training set
    full_stats  = train_target.groupby(train_series).agg(['mean', 'count'])
    full_stats['smoothed'] = (
        (full_stats['mean'] * full_stats['count'] + global_mean * smoothing)
        / (full_stats['count'] + smoothing)
    )
    test_encoded = test_series.map(full_stats['smoothed']).fillna(global_mean).values

    return train_encoded, test_encoded, full_stats['smoothed'].to_dict()


train_ne, test_ne, neighbourhood_means = kfold_target_encode(
    df.iloc[:TRAIN_SIZE]['neighbourhood_group'],
    df.iloc[:TRAIN_SIZE][TARGET],
    df.iloc[TRAIN_SIZE:]['neighbourhood_group'],
    n_splits=5, smoothing=5
)

df.loc[df.index[:TRAIN_SIZE], 'neighbourhood_encoded'] = train_ne
df.loc[df.index[TRAIN_SIZE:], 'neighbourhood_encoded'] = test_ne

print("Neighbourhood target encoding (smoothed mean log-price):")
for nb, val in sorted(neighbourhood_means.items(), key=lambda x: -x[1]):
    print(f"  {nb:20s}  →  {val:.4f}  (£{np.expm1(val):.0f}/night)")

In [ ]:
np.random.seed(42)

# ── Feature Inventory Table ──
# List every feature with type, null rate, and correlation with log_price

# Assemble all engineered numeric features into a single DataFrame
engineered_numeric_cols = [
    'bedrooms', 'bathrooms', 'accommodates', 'review_score', 'num_reviews',
    'host_experience_days', 'is_superhost', 'latitude', 'longitude',
    'num_reviews_log', 'host_experience_days_log', 'accommodates_log',
    'accommodates_per_bedroom', 'bathrooms_per_bedroom', 'review_credibility',
    'occupancy_proxy', 'host_tenure_years',
    'distance_to_centre_km', 'listing_density_1km',
    'superhost_x_review', 'bedrooms_x_accommodates', 'credibility_x_distance',
    'desc_length', 'has_luxury_words', 'has_budget_words',
    'neighbourhood_encoded',
]

# Only keep columns that actually exist in df
existing_cols = [c for c in engineered_numeric_cols if c in df.columns]

df_features = df[existing_cols + [TARGET]].copy()

inventory_rows = []
for col in existing_cols:
    origin = ('raw' if col in raw_feature_cols else 'engineered')
    null_rate = df_features[col].isnull().mean()
    corr_val  = df_features[col].fillna(df_features[col].median()).corr(df_features[TARGET])
    inventory_rows.append({
        'Feature':      col,
        'Origin':       origin,
        'Dtype':        str(df_features[col].dtype),
        'Null Rate (%)': round(null_rate * 100, 1),
        'Corr w/ target': round(corr_val, 3),
        '|Corr|':       round(abs(corr_val), 3),
    })

inventory_df = (
    pd.DataFrame(inventory_rows)
    .sort_values('|Corr|', ascending=False)
    .reset_index(drop=True)
)

print(f"Total features in inventory: {len(inventory_df)}")
print()
print(inventory_df.drop(columns='|Corr|').to_string(index=False))

In [ ]:
np.random.seed(42)

# ── Feature correlation overview: engineered vs raw ──
# Plot the top-15 most correlated features with log_price.
# Colour-code raw vs engineered to show the engineering contribution.

filled_df = df[existing_cols].fillna(df[existing_cols].median())
corr_with_target = filled_df.corrwith(df[TARGET]).abs().sort_values(ascending=False)
top15 = corr_with_target.head(15)

raw_mask = [f in raw_feature_cols for f in top15.index]
bar_colors = ['steelblue' if is_raw else 'darkorange' for is_raw in raw_mask]

fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(top15.index[::-1], top15.values[::-1], color=bar_colors[::-1], edgecolor='white')
ax.set_xlabel('|Pearson Correlation| with log_price')
ax.set_title('Top-15 Features by Correlation with Target')

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='steelblue', label='Raw feature'),
                   Patch(facecolor='darkorange', label='Engineered feature')]
ax.legend(handles=legend_elements, loc='lower right')
plt.tight_layout()
plt.show()

print("Engineered features in top-15:")
for f in top15.index:
    if f not in raw_feature_cols:
        print(f"  {f:35s} corr={corr_with_target[f]:.3f}")

In [ ]:
np.random.seed(42)

# ── Baseline model: Ridge on RAW 8 numeric features only ──
# This is the "before feature engineering" benchmark.

raw_numeric_baseline = [
    'bedrooms', 'bathrooms', 'accommodates', 'review_score',
    'num_reviews', 'host_experience_days', 'is_superhost', 'latitude', 'longitude'
]

# Build a minimal pipeline: impute → scale → Ridge
baseline_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
    ('model',   Ridge(alpha=1.0)),
])

X_base_train = df_train[raw_numeric_baseline].values
X_base_test  = df_test[raw_numeric_baseline].values

# 5-fold CV on training set
cv_base = cross_val_score(baseline_pipe, X_base_train, y_train.values,
                          cv=5, scoring='r2', n_jobs=-1)
cv_rmse_base = cross_val_score(baseline_pipe, X_base_train, y_train.values,
                               cv=5, scoring='neg_root_mean_squared_error', n_jobs=-1)

# Fit and evaluate on held-out test set
baseline_pipe.fit(X_base_train, y_train.values)
y_pred_base   = baseline_pipe.predict(X_base_test)
test_r2_base  = r2_score(y_test.values, y_pred_base)
test_rmse_base_log = np.sqrt(mean_squared_error(y_test.values, y_pred_base))  # in log space
test_rmse_base_gbp = np.sqrt(mean_squared_error(
    np.expm1(y_test.values), np.expm1(y_pred_base)
))

print("=" * 55)
print("BASELINE MODEL — Ridge on 9 raw numeric features")
print("=" * 55)
print(f"  CV R²              : {cv_base.mean():.4f} ± {cv_base.std():.4f}")
print(f"  CV RMSE (log)      : {(-cv_rmse_base).mean():.4f} ± {(-cv_rmse_base).std():.4f}")
print(f"  Test R²            : {test_r2_base:.4f}")
print(f"  Test RMSE (log)    : {test_rmse_base_log:.4f}")
print(f"  Test RMSE (£)      : £{test_rmse_base_gbp:.2f}")
print("=" * 55)

r2_base = cv_base.mean()  # store for comparison

In [ ]:
np.random.seed(42)

# ── Engineered model: Ridge on ALL engineered features ──
# We use the existing_cols list from the inventory, impute and scale.

X_eng_train = df.iloc[:TRAIN_SIZE][existing_cols].values
X_eng_test  = df.iloc[TRAIN_SIZE:][existing_cols].values

engineered_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
    ('model',   Ridge(alpha=1.0)),
])

cv_eng = cross_val_score(engineered_pipe, X_eng_train, y_train.values,
                         cv=5, scoring='r2', n_jobs=-1)
cv_rmse_eng = cross_val_score(engineered_pipe, X_eng_train, y_train.values,
                              cv=5, scoring='neg_root_mean_squared_error', n_jobs=-1)

engineered_pipe.fit(X_eng_train, y_train.values)
y_pred_eng     = engineered_pipe.predict(X_eng_test)
test_r2_eng    = r2_score(y_test.values, y_pred_eng)
test_rmse_eng_gbp = np.sqrt(mean_squared_error(
    np.expm1(y_test.values), np.expm1(y_pred_eng)
))

r2_eng = cv_eng.mean()
delta_r2 = r2_eng - r2_base

print("=" * 55)
print("ENGINEERED MODEL — Ridge on all engineered features")
print("=" * 55)
print(f"  CV R²              : {r2_eng:.4f} ± {cv_eng.std():.4f}")
print(f"  CV RMSE (log)      : {(-cv_rmse_eng).mean():.4f} ± {(-cv_rmse_eng).std():.4f}")
print(f"  Test R²            : {test_r2_eng:.4f}")
print(f"  Test RMSE (£)      : £{test_rmse_eng_gbp:.2f}")
print()
print(f"  Δ R² vs. baseline  : +{delta_r2:.1%}")
print("=" * 55)

# Assert that feature engineering gives ≥ 5% R² improvement
assert delta_r2 >= 0.05, (
    f"Feature engineering improvement {delta_r2:.1%} is below the 5% target."
    " Check feature construction."
)
print(f"\nAssertion passed: Δ R² = {delta_r2:.1%} ≥ 5%")

In [ ]:
np.random.seed(42)

# ── Feature selection: Mutual Information ranking ──
# Mutual Information measures how much knowing feature X reduces uncertainty about y.
# It captures non-linear relationships that correlation misses.

X_eng_imputed = SimpleImputer(strategy='median').fit_transform(X_eng_train)

mi_scores = mutual_info_regression(
    X_eng_imputed, y_train.values,
    random_state=42
)

mi_df = (
    pd.DataFrame({'feature': existing_cols, 'mi_score': mi_scores})
    .sort_values('mi_score', ascending=False)
    .reset_index(drop=True)
)

TOP_N = 25
top_features = mi_df.head(TOP_N)['feature'].tolist()

# Horizontal bar chart of top-25 MI scores
fig, ax = plt.subplots(figsize=(9, 8))
colors = ['steelblue' if f in raw_feature_cols else 'darkorange'
          for f in mi_df.head(TOP_N)['feature']]
ax.barh(
    mi_df.head(TOP_N)['feature'][::-1],
    mi_df.head(TOP_N)['mi_score'][::-1],
    color=colors[::-1], edgecolor='white'
)
ax.set_xlabel('Mutual Information Score')
ax.set_title('Top-25 Features by Mutual Information\n(blue = raw, orange = engineered)')
# Legend patches
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='steelblue', label='Raw feature'),
                   Patch(facecolor='darkorange', label='Engineered feature')]
ax.legend(handles=legend_elements, loc='lower right')
plt.tight_layout()
plt.show()

print(f"Top-{TOP_N} features by Mutual Information:\n{top_features}")

In [ ]:
np.random.seed(42)

# ── Lasso feature selection on the top-25 MI features ──
# Lasso (L1 regularisation) shrinks unimportant coefficients to exactly zero.
# The surviving non-zero features form our final modelling feature set.

from sklearn.linear_model import Lasso

# Subset to top-25 MI features
top25_idx   = [existing_cols.index(f) for f in top_features]
X_top25     = X_eng_imputed[:, top25_idx]

lasso_selector = Pipeline([
    ('scaler', StandardScaler()),
    ('lasso',  Lasso(alpha=0.001, max_iter=5000, random_state=42)),
])
lasso_selector.fit(X_top25, y_train.values)

lasso_coefs   = lasso_selector.named_steps['lasso'].coef_
selected_mask = lasso_coefs != 0.0
selected_features = [f for f, keep in zip(top_features, selected_mask) if keep]

print(f"Lasso retained {selected_mask.sum()} / {TOP_N} features")
print()
print("Selected features and |coefficient|:")
for feat, coef in sorted(
    zip(top_features, lasso_coefs), key=lambda x: -abs(x[1])
):
    status = 'KEEP' if coef != 0 else 'drop'
    print(f"  {status:4s}  {feat:35s}  coef={coef:+.4f}")

In [ ]:
np.random.seed(42)

# ── Feature engineering impact visualisation ──
# Three-bar chart: baseline, all-engineered, Lasso-selected

# Compute CV R² for Lasso-selected feature set
sel_idx     = [existing_cols.index(f) for f in selected_features]
X_selected  = X_eng_train[:, sel_idx]  # raw (not imputed) — let pipeline handle it
X_sel_test  = X_eng_test[:, sel_idx]

selected_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
    ('model',   Ridge(alpha=1.0)),
])
cv_sel = cross_val_score(selected_pipe, X_selected, y_train.values,
                         cv=5, scoring='r2', n_jobs=-1)
r2_sel = cv_sel.mean()

# Bar chart
labels = ['Baseline\n(9 raw)', 'All Engineered\n(26 features)', f'Lasso Selected\n({len(selected_features)} features)']
values = [r2_base, r2_eng, r2_sel]
colors = ['#d62728', '#ff7f0e', '#2ca02c']

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(labels, values, color=colors, width=0.5, edgecolor='white')
ax.set_ylim(0, 1.0)
ax.set_ylabel('Cross-Validated R²')
ax.set_title('Feature Engineering Impact on Model Performance (Ridge)')

for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width() / 2, val + 0.01,
            f'{val:.3f}', ha='center', va='bottom', fontweight='bold')

# Annotate improvement arrows
ax.annotate(
    f'+{r2_eng - r2_base:.1%} vs baseline',
    xy=(1, r2_eng), xytext=(1.4, r2_eng + 0.05),
    arrowprops=dict(arrowstyle='->', color='black'), fontsize=9
)

plt.tight_layout()
plt.show()

print(f"R² improvement (all engineered vs baseline): {(r2_eng - r2_base):.1%}")
print(f"R² improvement (Lasso selected vs baseline): {(r2_sel - r2_base):.1%}")

## Part B — Project Milestone 2: 5-Algorithm Comparison

We now move from feature engineering (Ridge baseline) to a full algorithm comparison.

We will evaluate 5 algorithms on the Lasso-selected feature set using **5-fold cross-validation**:
1. Ridge (linear, regularised)
2. RandomForest (bagging ensemble)
3. GradientBoosting (boosting ensemble)
4. SVR (kernel-based)
5. XGBoost (if installed) or ExtraTrees (fallback)

Metrics reported in **original £ scale** (after applying `expm1`) so they are interpretable by the business team.


In [ ]:
np.random.seed(42)

# ── Define 5 algorithms ──
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor
from sklearn.linear_model import Ridge
from sklearn.svm import SVR

models = {
    'Ridge':           Ridge(alpha=1.0),
    'RandomForest':    RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'GradientBoosting': GradientBoostingRegressor(n_estimators=100, random_state=42),
    'SVR':             SVR(kernel='rbf', C=10.0),
}

try:
    from xgboost import XGBRegressor
    models['XGBoost'] = XGBRegressor(
        n_estimators=100, random_state=42,
        verbosity=0, n_jobs=-1
    )
    print("XGBoost available — included in comparison.")
except ImportError:
    models['ExtraTrees'] = ExtraTreesRegressor(n_estimators=100, random_state=42, n_jobs=-1)
    print("XGBoost not installed — using ExtraTreesRegressor as 5th algorithm.")
    print("  Install with: pip install xgboost")

print(f"\nAlgorithms in comparison: {list(models.keys())}")

In [ ]:
np.random.seed(42)

# ── 5-fold CV for all models on the Lasso-selected features ──
# Metrics are computed in log space for CV, then converted to £ for reporting.

# Prepare scaled feature matrix once (shared across all models)
preprocess_for_models = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
])
X_sel_scaled       = preprocess_for_models.fit_transform(X_selected)
X_sel_test_scaled  = preprocess_for_models.transform(X_sel_test)

benchmark_rows = []
kf5 = KFold(n_splits=5, shuffle=True, random_state=42)

for name, model in models.items():
    cv_r2   = cross_val_score(model, X_sel_scaled, y_train.values, cv=kf5, scoring='r2')
    cv_rmse = cross_val_score(model, X_sel_scaled, y_train.values,
                              cv=kf5, scoring='neg_root_mean_squared_error')
    cv_mae  = cross_val_score(model, X_sel_scaled, y_train.values,
                              cv=kf5, scoring='neg_mean_absolute_error')

    # Fit on full training set for test-set metrics
    model.fit(X_sel_scaled, y_train.values)
    y_pred_log = model.predict(X_sel_test_scaled)
    test_rmse_gbp = np.sqrt(mean_squared_error(
        np.expm1(y_test.values), np.expm1(y_pred_log)
    ))
    test_r2  = r2_score(y_test.values, y_pred_log)
    test_mae_gbp = mean_absolute_error(
        np.expm1(y_test.values), np.expm1(y_pred_log)
    )

    # CV RMSE converted to approximate £ scale (rough: exp mean log-RMSE)
    cv_rmse_mean_log = (-cv_rmse).mean()

    benchmark_rows.append({
        'Algorithm':   name,
        'CV R²':       round(cv_r2.mean(), 4),
        'CV R² std':   round(cv_r2.std(), 4),
        'CV RMSE (log)': round(cv_rmse_mean_log, 4),
        'Test R²':     round(test_r2, 4),
        'Test RMSE (£)': round(test_rmse_gbp, 2),
        'Test MAE (£)': round(test_mae_gbp, 2),
    })
    print(f"{name:20s}  CV R²={cv_r2.mean():.4f}  Test RMSE=£{test_rmse_gbp:.2f}")

benchmark_df = pd.DataFrame(benchmark_rows).sort_values('Test RMSE (£)')
print()
print(benchmark_df.to_string(index=False))

best_model_name   = benchmark_df.iloc[0]['Algorithm']
best_model_object = models[best_model_name]
print(f"\nBest algorithm: {best_model_name}")

In [ ]:
np.random.seed(42)

# ── Hyperparameter tuning for the best model ──
# Try Optuna for Bayesian optimisation; fall back to GridSearchCV.

# We tune GradientBoostingRegressor parameters directly
# (or XGBoost if that was the best model).

best_params = {}

try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)

    def objective(trial):
        params = {
            'n_estimators':  trial.suggest_int('n_estimators',  50, 300),
            'max_depth':     trial.suggest_int('max_depth',      3,   8),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
            'subsample':     trial.suggest_float('subsample',     0.6, 1.0),
            'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 20),
            'random_state':  42,
        }
        model_trial = GradientBoostingRegressor(**params)
        scores = cross_val_score(
            model_trial, X_sel_scaled, y_train.values,
            cv=3, scoring='neg_root_mean_squared_error'
        )
        return (-scores).mean()  # minimise RMSE

    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=25, show_progress_bar=False)
    best_params = study.best_params
    best_params['random_state'] = 42
    print("Optuna tuning complete.")
    print(f"Best params: {best_params}")
    print(f"Best CV RMSE (log): {study.best_value:.4f}")

except ImportError:
    print("Optuna not installed — using GridSearchCV.")
    print("Install with: pip install optuna")
    from sklearn.model_selection import GridSearchCV
    param_grid = {
        'n_estimators':  [100, 200],
        'max_depth':     [3, 5],
        'learning_rate': [0.05, 0.1],
    }
    gs = GridSearchCV(
        GradientBoostingRegressor(random_state=42),
        param_grid, cv=3,
        scoring='neg_root_mean_squared_error',
        n_jobs=-1, verbose=0
    )
    gs.fit(X_sel_scaled, y_train.values)
    best_params = gs.best_params_
    best_params['random_state'] = 42
    print(f"Best params: {best_params}")
    print(f"Best CV RMSE (log): {-gs.best_score_:.4f}")

In [ ]:
np.random.seed(42)

# ── Fit best tuned model on full training set ──
tuned_model = GradientBoostingRegressor(**best_params)
tuned_model.fit(X_sel_scaled, y_train.values)

y_pred_tuned_log = tuned_model.predict(X_sel_test_scaled)
y_pred_tuned_gbp = np.expm1(y_pred_tuned_log)
y_test_gbp       = np.expm1(y_test.values)

test_rmse_tuned = np.sqrt(mean_squared_error(y_test_gbp, y_pred_tuned_gbp))
test_mae_tuned  = mean_absolute_error(y_test_gbp, y_pred_tuned_gbp)
test_r2_tuned   = r2_score(y_test.values, y_pred_tuned_log)

RULE_BASED_RMSE = 42.0  # £42 from the rule-based system

print("=" * 55)
print("TUNED MODEL — GradientBoostingRegressor")
print("=" * 55)
print(f"  Test R²               : {test_r2_tuned:.4f}")
print(f"  Test RMSE             : £{test_rmse_tuned:.2f}")
print(f"  Test MAE              : £{test_mae_tuned:.2f}")
print()
print(f"  Rule-based RMSE       : £{RULE_BASED_RMSE:.2f}")
print(f"  ML improvement        : {((RULE_BASED_RMSE - test_rmse_tuned) / RULE_BASED_RMSE):.1%} reduction in RMSE")
print()

if test_rmse_tuned < 30:
    print("TARGET ACHIEVED: RMSE < £30 — ML system ready for production!")
else:
    print(f"RMSE = £{test_rmse_tuned:.2f} — additional tuning recommended to reach < £30.")
print("=" * 55)

In [ ]:
np.random.seed(42)

# ── Residual analysis — 2×2 diagnostic figure ──
fig, axes = plt.subplots(2, 2, figsize=(13, 10))

# (a) Predicted vs. Actual (log scale)
ax = axes[0, 0]
ax.scatter(y_test.values, y_pred_tuned_log, alpha=0.3, s=10, color='steelblue')
lims = [min(y_test.min(), y_pred_tuned_log.min()),
        max(y_test.max(), y_pred_tuned_log.max())]
ax.plot(lims, lims, 'r--', lw=1.5, label='Perfect prediction')
ax.set_xlabel('Actual log(price)')
ax.set_ylabel('Predicted log(price)')
ax.set_title(f'(a) Predicted vs. Actual  (R²={test_r2_tuned:.3f})')
ax.legend()

# (b) Residuals vs. Predicted
ax = axes[0, 1]
residuals = y_test.values - y_pred_tuned_log
ax.scatter(y_pred_tuned_log, residuals, alpha=0.3, s=10, color='darkorange')
ax.axhline(0, color='red', linestyle='--', lw=1.5)
ax.set_xlabel('Predicted log(price)')
ax.set_ylabel('Residual')
ax.set_title('(b) Residuals vs. Predicted')

# (c) Residual histogram with normal overlay
ax = axes[1, 0]
ax.hist(residuals, bins=40, density=True, color='mediumseagreen', edgecolor='white', alpha=0.8)
from scipy import stats
x_norm = np.linspace(residuals.min(), residuals.max(), 200)
ax.plot(x_norm, stats.norm.pdf(x_norm, residuals.mean(), residuals.std()),
        'r-', lw=2, label='Normal fit')
ax.set_xlabel('Residual')
ax.set_ylabel('Density')
ax.set_title('(c) Residual Distribution')
ax.legend()

# (d) Feature importance bar chart (top 15)
ax = axes[1, 1]
importances = tuned_model.feature_importances_
top15_idx   = np.argsort(importances)[-15:]
top15_names = [selected_features[i] for i in top15_idx]
top15_vals  = importances[top15_idx]
ax.barh(top15_names, top15_vals, color='slateblue', edgecolor='white')
ax.set_xlabel('Feature Importance (impurity reduction)')
ax.set_title('(d) Top-15 Feature Importances')

plt.suptitle('Tuned GradientBoosting — Residual Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
np.random.seed(42)

# ── Final benchmark table with styling ──
# Add the tuned model to the benchmark DataFrame

tuned_row = pd.DataFrame([{
    'Algorithm':       'GBT (tuned)',
    'CV R²':           round(cross_val_score(
                           tuned_model, X_sel_scaled, y_train.values, cv=5, scoring='r2'
                       ).mean(), 4),
    'CV R² std':       round(cross_val_score(
                           tuned_model, X_sel_scaled, y_train.values, cv=5, scoring='r2'
                       ).std(), 4),
    'CV RMSE (log)':   round((-cross_val_score(
                           tuned_model, X_sel_scaled, y_train.values,
                           cv=5, scoring='neg_root_mean_squared_error'
                       )).mean(), 4),
    'Test R²':         round(test_r2_tuned, 4),
    'Test RMSE (£)':   round(test_rmse_tuned, 2),
    'Test MAE (£)':    round(test_mae_tuned, 2),
}])

full_benchmark = pd.concat([benchmark_df, tuned_row], ignore_index=True)
full_benchmark = full_benchmark.sort_values('Test RMSE (£)').reset_index(drop=True)

try:
    # Pandas styling for Jupyter display
    from IPython.display import display
    styled = (
        full_benchmark.style
        .highlight_min(subset=['Test RMSE (£)', 'CV RMSE (log)'], axis=0, color='#d4edda')
        .highlight_max(subset=['Test R²', 'CV R²'],               axis=0, color='#d4edda')
        .set_caption('Algorithm Benchmark — PropInsight Pricing System')
        .format({
            'CV R²': '{:.4f}', 'Test R²': '{:.4f}',
            'Test RMSE (£)': '£{:.2f}', 'Test MAE (£)': '£{:.2f}'
        })
    )
    display(styled)
except Exception:
    print(full_benchmark.to_string(index=False))

print(f"\nRule-based baseline RMSE  : £{RULE_BASED_RMSE:.2f}")
print(f"Best ML model RMSE        : £{full_benchmark['Test RMSE (£)'].min():.2f}")

## Part B — Production Pipeline

We now package everything into a single sklearn `Pipeline` that can be shipped to production.  
The pipeline accepts **raw listing dictionaries** and returns **price predictions** — no external preprocessing needed.


In [ ]:
np.random.seed(42)

# ── Full production pipeline with ColumnTransformer ──
# Accepts raw DataFrame with the 12 original columns.

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.feature_extraction.text import TfidfVectorizer

numeric_cols = [
    'bedrooms', 'bathrooms', 'accommodates', 'review_score',
    'num_reviews', 'host_experience_days', 'is_superhost', 'latitude', 'longitude'
]
cat_cols = ['room_type', 'neighbourhood_group']
text_col = 'description'

preprocessor_prod = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('imp', SimpleImputer(strategy='median')),
            ('scl', StandardScaler()),
        ]), numeric_cols),
        ('cat', Pipeline([
            ('imp', SimpleImputer(strategy='most_frequent')),
            ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
        ]), cat_cols),
        ('txt', TfidfVectorizer(max_features=50, stop_words='english'), text_col),
    ],
    remainder='drop',
    n_jobs=-1
)

# Use the best tuned parameters for the production model
prod_model = GradientBoostingRegressor(**best_params)

production_pipeline = Pipeline([
    ('preprocess', preprocessor_prod),
    ('model',      prod_model),
])

# Fit on the raw training DataFrame (no manual feature engineering needed)
production_pipeline.fit(X_train_raw, y_train)

y_prod_pred = production_pipeline.predict(X_test_raw)
prod_rmse   = np.sqrt(mean_squared_error(
    np.expm1(y_test.values), np.expm1(y_prod_pred)
))
prod_r2     = r2_score(y_test.values, y_prod_pred)

print(f"Production pipeline fitted.")
print(f"Input  : raw DataFrame with {len(numeric_cols + cat_cols) + 1} columns")
print(f"Output : log-price prediction (apply expm1 for £)")
print(f"Test RMSE (£) : £{prod_rmse:.2f}")
print(f"Test R²       : {prod_r2:.4f}")

In [ ]:
np.random.seed(42)

import joblib

# ── Serialise and verify the production pipeline ──
pipeline_path = 'propinsight_v1.pkl'
joblib.dump(production_pipeline, pipeline_path)

file_size_kb = os.path.getsize(pipeline_path) / 1024
print(f"Pipeline serialised to '{pipeline_path}' ({file_size_kb:.1f} KB)")

# Reload from disk
loaded_pipeline = joblib.load(pipeline_path)

# Verify predictions are numerically identical
original_preds = production_pipeline.predict(X_test_raw.head(3))
loaded_preds   = loaded_pipeline.predict(X_test_raw.head(3))

np.testing.assert_array_almost_equal(original_preds, loaded_preds, decimal=8)
print("Pipeline serialized and verified — predictions are bit-for-bit identical.")

print()
for i, (orig, load) in enumerate(zip(original_preds, loaded_preds)):
    print(f"  Row {i}: log_price={orig:.5f}  →  £{np.expm1(orig):.2f}/night")

In [ ]:
np.random.seed(42)

# ── REST prediction endpoint simulation ──
# In production this function would be a Flask/FastAPI route decorated with @app.post('/predict').
# Here we simulate the full request-response lifecycle.

def predict_endpoint(request_json):
    """Simulate a REST prediction endpoint.

    Parameters
    ----------
    request_json : dict
        JSON body with listing features.

    Returns
    -------
    dict
        JSON response with prediction and confidence interval.
    """
    pipeline   = joblib.load('propinsight_v1.pkl')        # production: load once at startup
    df_input   = pd.DataFrame([request_json])             # single-row DataFrame
    log_price  = pipeline.predict(df_input)[0]            # predict in log space
    price_gbp  = np.expm1(log_price)                      # convert to £

    # Approximate 90% CI using ±0.25 in log space (corresponds to ~±28% in £)
    ci_low  = np.expm1(log_price - 0.25)
    ci_high = np.expm1(log_price + 0.25)

    return {
        'status':                  'success',
        'predicted_price_gbp':     round(float(price_gbp), 2),
        'confidence_interval_90':  [round(float(ci_low), 2), round(float(ci_high), 2)],
        'model_version':           'propinsight_v1',
    }


# Test with 3 representative listing types
test_listings_api = [
    {
        'bedrooms': 2, 'bathrooms': 1.0, 'accommodates': 4,
        'review_score': 4.8, 'num_reviews': 45,
        'host_experience_days': 1200, 'is_superhost': 1,
        'room_type': 'Entire home/apt', 'neighbourhood_group': 'Westminster',
        'latitude': 51.505, 'longitude': -0.120,
        'description': 'Beautiful apartment with panoramic city views rooftop',
    },
    {
        'bedrooms': 1, 'bathrooms': 0.5, 'accommodates': 2,
        'review_score': 3.9, 'num_reviews': 8,
        'host_experience_days': 180, 'is_superhost': 0,
        'room_type': 'Private room', 'neighbourhood_group': 'Hackney',
        'latitude': 51.545, 'longitude': -0.055,
        'description': 'Budget room shared bathroom close to centre',
    },
    {
        'bedrooms': 4, 'bathrooms': 3.0, 'accommodates': 8,
        'review_score': 4.95, 'num_reviews': 210,
        'host_experience_days': 2500, 'is_superhost': 1,
        'room_type': 'Entire home/apt', 'neighbourhood_group': 'Westminster',
        'latitude': 51.501, 'longitude': -0.125,
        'description': 'luxury penthouse rooftop terrace chef kitchen skyline',
    },
]

print("REST Endpoint — /predict\n" + "=" * 40)
for i, listing in enumerate(test_listings_api, 1):
    response = predict_endpoint(listing)
    print(f"\nListing {i}: {listing['room_type']}, {listing['neighbourhood_group']}")
    print(json.dumps(response, indent=2))

In [ ]:
np.random.seed(42)

# ── Model Card ──
import sklearn

cv_rmse_prod = (-cross_val_score(
    production_pipeline, X_train_raw, y_train,
    cv=5, scoring='neg_root_mean_squared_error'
)).mean()

# Convert log-RMSE to approximate £ RMSE via exp(log_rmse) approximation
pct_improvement = ((RULE_BASED_RMSE - prod_rmse) / RULE_BASED_RMSE) * 100

model_card = f"""
╔══════════════════════════════════════════════════════════════════╗
║         MODEL CARD — PropInsight Price Predictor v1.0           ║
╠══════════════════════════════════════════════════════════════════╣
║ Model type       : GradientBoostingRegressor (sklearn {sklearn.__version__:6s})    ║
║ Training data    : 4,000 synthetic Airbnb-style listings         ║
║ Features         : 12 raw → preprocessed via ColumnTransformer  ║
║                    (numeric + categorical + TF-IDF text)         ║
╠══════════════════════════════════════════════════════════════════╣
║ PERFORMANCE                                                      ║
║   CV RMSE (log)  : {cv_rmse_prod:.4f}                                    ║
║   Test RMSE (£)  : £{prod_rmse:.2f}                                    ║
║   Test R²        : {prod_r2:.4f}                                    ║
║   vs. Rule-based : £{RULE_BASED_RMSE:.0f} RMSE → £{prod_rmse:.1f} RMSE ({pct_improvement:.0f}% improvement) ║
╠══════════════════════════════════════════════════════════════════╣
║ INTENDED USE                                                     ║
║   Nightly price estimation for property hosts on short-term      ║
║   rental platforms. Provides a data-driven starting point for    ║
║   pricing decisions; not a binding recommendation.               ║
╠══════════════════════════════════════════════════════════════════╣
║ NOT FOR                                                          ║
║   Legal or regulatory pricing decisions.                         ║
║   Discriminatory pricing based on protected characteristics.     ║
║   Areas significantly outside the training data distribution.    ║
╠══════════════════════════════════════════════════════════════════╣
║ KNOWN LIMITATIONS                                                ║
║   1. Trained on synthetic data — production retraining required  ║
║   2. May underestimate luxury properties (sparse in training)    ║
║   3. Geographic encoding limited to 8 London boroughs            ║
║   4. No seasonality or demand signals encoded                    ║
╚══════════════════════════════════════════════════════════════════╝
"""
print(model_card)

In [ ]:
np.random.seed(42)

# ── Data drift monitoring ──
# In production, new batches of listings may have different statistical properties
# (e.g., due to market shifts, seasonal patterns, or data collection changes).
# We compute a Z-score for each numeric feature's mean relative to training statistics.
# A Z > 3 flags a statistically significant drift.

def build_training_stats(X_df, numeric_columns):
    """Compute mean and std for each numeric column from training data."""
    stats = {}
    for col in numeric_columns:
        if col in X_df.columns:
            stats[col] = {
                'mean': X_df[col].mean(),
                'std':  X_df[col].std(),
            }
    return stats


def check_drift(new_data_df, training_stats_dict, z_threshold=3.0):
    """Compare new batch statistics to training statistics.

    Returns a list of alert strings, or a single no-drift message.
    """
    alerts = []
    for col, stats in training_stats_dict.items():
        if col not in new_data_df.columns:
            continue
        new_mean = new_data_df[col].mean()
        z_score  = abs(new_mean - stats['mean']) / (stats['std'] + 1e-10)
        status   = 'DRIFT' if z_score > z_threshold else 'OK   '
        alerts.append(f"  {status}  {col:30s}  train_mean={stats['mean']:.3f}  "
                      f"new_mean={new_mean:.3f}  Z={z_score:.2f}")
    return alerts


training_stats = build_training_stats(X_train_raw, numeric_cols)

# Simulate three scenarios
print("Scenario 1: Normal new batch (no drift expected)")
normal_batch = X_test_raw.copy()
for line in check_drift(normal_batch, training_stats): print(line)

print("\nScenario 2: Simulated drift — num_reviews inflated 5×")
drifted_batch = X_test_raw.copy()
drifted_batch['num_reviews'] = drifted_batch['num_reviews'] * 5
for line in check_drift(drifted_batch, training_stats): print(line)

print("\nScenario 3: Simulated drift — bedrooms shifted +2")
shifted_batch = X_test_raw.copy()
shifted_batch['bedrooms'] = shifted_batch['bedrooms'] + 2
for line in check_drift(shifted_batch, training_stats): print(line)

In [ ]:
np.random.seed(42)

# ── Partial dependence plots (PDPs) for top 4 features ──
# PDPs show the marginal effect of a single feature on the prediction,
# averaging out all other features.

from sklearn.inspection import PartialDependenceDisplay

# We need a fitted sklearn estimator with a numpy array interface.
# Use the scaled+imputed X_sel_scaled and the tuned_model already fitted.

# Pick the 4 most important features by name (from the Lasso selected set)
importances_arr = tuned_model.feature_importances_
top4_idx = np.argsort(importances_arr)[-4:][::-1]
top4_names = [selected_features[i] for i in top4_idx]

print(f"Plotting PDPs for: {top4_names}")

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for ax, feat_name, feat_idx in zip(axes, top4_names, top4_idx):
    # Manually compute PDP by sweeping feature value
    X_sweep = X_sel_scaled.copy()
    col_vals = np.linspace(
        np.percentile(X_sel_scaled[:, feat_idx], 2),
        np.percentile(X_sel_scaled[:, feat_idx], 98),
        50
    )
    pdp_vals = []
    for v in col_vals:
        X_tmp = X_sweep.copy()
        X_tmp[:, feat_idx] = v              # set all rows to this value
        pdp_vals.append(tuned_model.predict(X_tmp).mean())  # marginal prediction

    # Convert x-axis back to original scale for interpretability (approximate)
    ax.plot(col_vals, pdp_vals, color='steelblue', lw=2)
    ax.set_xlabel(f'{feat_name}\n(standardised)')
    ax.set_ylabel('Partial dependence\n(log price)')
    ax.set_title(feat_name, fontsize=9)

plt.suptitle('Partial Dependence Plots — Top-4 Features', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
np.random.seed(42)

# ── Price prediction error by neighbourhood ──
# Understand where the model struggles — systematic bias by location?

test_with_preds = X_test_raw.copy()
test_with_preds['actual_log_price']    = y_test.values
test_with_preds['predicted_log_price'] = y_prod_pred
test_with_preds['abs_error_gbp'] = np.abs(
    np.expm1(y_test.values) - np.expm1(y_prod_pred)
)

neighbourhood_error = (
    test_with_preds.groupby('neighbourhood_group')['abs_error_gbp']
    .agg(['mean', 'median', 'count'])
    .sort_values('mean', ascending=False)
    .reset_index()
)
neighbourhood_error.columns = ['Neighbourhood', 'Mean |Error| (£)', 'Median |Error| (£)', 'Count']

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(
    neighbourhood_error['Neighbourhood'],
    neighbourhood_error['Mean |Error| (£)'],
    color='coral', edgecolor='white'
)
ax.axhline(neighbourhood_error['Mean |Error| (£)'].mean(),
           color='steelblue', linestyle='--', lw=1.5, label='Overall mean MAE')
ax.set_xlabel('Neighbourhood')
ax.set_ylabel('Mean Absolute Error (£)')
ax.set_title('Prediction Error by Neighbourhood — Where does the model struggle?')
ax.tick_params(axis='x', rotation=30)
ax.legend()
plt.tight_layout()
plt.show()

print(neighbourhood_error.to_string(index=False))

In [ ]:
np.random.seed(42)

# ── Price prediction error by room type ──

room_error = (
    test_with_preds.groupby('room_type')['abs_error_gbp']
    .agg(['mean', 'median', 'count'])
    .sort_values('mean', ascending=False)
    .reset_index()
)
room_error.columns = ['Room Type', 'Mean |Error| (£)', 'Median |Error| (£)', 'Count']

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart of error by room type
ax = axes[0]
ax.bar(room_error['Room Type'], room_error['Mean |Error| (£)'],
       color=['#2ca02c', '#ff7f0e', '#d62728'])
ax.set_ylabel('Mean |Error| (£)')
ax.set_title('Error by Room Type')

# Actual vs. predicted by room type
ax = axes[1]
room_colours = {'Entire home/apt': '#2ca02c', 'Private room': '#ff7f0e', 'Shared room': '#d62728'}
sample_idx = np.random.choice(len(test_with_preds), 300, replace=False)
for room in test_with_preds['room_type'].unique():
    mask = test_with_preds['room_type'].iloc[sample_idx] == room
    ax.scatter(
        test_with_preds['actual_log_price'].iloc[sample_idx][mask],
        test_with_preds['predicted_log_price'].iloc[sample_idx][mask],
        alpha=0.4, s=15, label=room, color=room_colours.get(room, 'grey')
    )
lims = [test_with_preds['actual_log_price'].min(), test_with_preds['actual_log_price'].max()]
ax.plot(lims, lims, 'k--', lw=1)
ax.set_xlabel('Actual log(price)')
ax.set_ylabel('Predicted log(price)')
ax.set_title('Predicted vs. Actual by Room Type')
ax.legend(fontsize=8)

plt.suptitle('Disaggregated Model Performance', fontweight='bold')
plt.tight_layout()
plt.show()

print(room_error.to_string(index=False))

In [ ]:
np.random.seed(42)

# ── Learning curve: how does model performance scale with training size? ──
# Useful for deciding whether to collect more data.

from sklearn.model_selection import learning_curve

train_sizes, train_scores, val_scores = learning_curve(
    production_pipeline,
    X_train_raw, y_train,
    train_sizes=np.linspace(0.1, 1.0, 8),
    cv=4,
    scoring='r2',
    n_jobs=-1,
    random_state=42
)

train_mean = train_scores.mean(axis=1)
val_mean   = val_scores.mean(axis=1)
val_std    = val_scores.std(axis=1)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(train_sizes, train_mean, 'o-', color='darkorange', label='Training R²')
ax.plot(train_sizes, val_mean,   'o-', color='steelblue',  label='CV Validation R²')
ax.fill_between(train_sizes,
                val_mean - val_std,
                val_mean + val_std,
                alpha=0.15, color='steelblue')
ax.set_xlabel('Training set size (rows)')
ax.set_ylabel('R²')
ax.set_title('Learning Curve — Production Pipeline')
ax.legend()
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

# Interpretation
if train_mean[-1] - val_mean[-1] > 0.15:
    print("High variance: training and validation gap > 0.15 — model may be overfitting.")
    print("Consider: regularisation, more data, feature reduction.")
elif val_mean[-1] < 0.5:
    print("High bias: validation R² < 0.5 — model is underfitting.")
    print("Consider: richer features, less regularisation, more complex model.")
else:
    print(f"Healthy learning curve. Validation R² at full training size: {val_mean[-1]:.4f}")

## Retrospective Questions

Reflect on the choices made in this lab before reading the answers.

---

**Q1.** Which single engineered feature contributed most to the R² improvement, and why does it outperform the raw features that went into it?

<details><summary>Show answer</summary>

In most runs `review_credibility = review_score × log1p(num_reviews)` ranks among the top 2–3 features by mutual information. Here is why it outperforms either raw feature alone:

- A 4.9 rating from 2 reviews is less meaningful than a 4.7 rating from 300 reviews.
- `review_score` alone doesn't encode confidence. `num_reviews` alone doesn't encode quality.
- Their product captures *credibility*: a listing must be consistently good (high score) **and** well-visited (many reviews) to score highly. This joint signal has a higher linear correlation with price than either component.
- The `log1p` dampens the influence of extremely prolific listings (1000 reviews) without discarding the signal entirely.

This is a general lesson: **multiplicative interactions often encode domain knowledge that additive linear models cannot recover from raw features**.

</details>

---

**Q2.** Why did we use a temporal train/test split (first 4000 rows = train) rather than a random split?

<details><summary>Show answer</summary>

The synthetic dataset simulates the **temporal ordering** of real Airbnb listings (newer listings appear later). A random split would mix future rows into the training set, creating temporal leakage:

- **Target encoding** would use future listing statistics to encode past listings.
- **Geographic density** features (BallTree) would include future listings in the neighbourhood count.
- **Market trends** (if present) would leak from future to past through the split.

A temporal split gives a more realistic estimate of how well the model will perform on truly **unseen future listings** — which is what matters in production. Random splits consistently overestimate performance for time-ordered data.

</details>

---

**Q3.** What would change if you had 500,000 listings instead of 5,000? Name at least three scaling considerations.

<details><summary>Show answer</summary>

1. **BallTree geographic density:** The pairwise distance computation is O(n log n) at query time. At 500k listings, querying all rows becomes slow. Replace with spatial indexing (H3 hexagonal grids, PostGIS, or approximate nearest neighbour libraries like `annoy`).

2. **Target encoding:** K-fold target encoding requires cycling through the data n_splits times. At 500k rows with 5 folds this is still manageable, but the groupby operations become memory-intensive. Switch to hash-based encoders or a streaming variant.

3. **Gradient Boosting training time:** GBT scales poorly with n — training time is O(n × trees × depth). At 500k rows, switch to histogram-based implementations (`HistGradientBoostingRegressor`, `LightGBM`) which bin continuous features and achieve near-linear scaling.

4. **Cross-validation:** 5-fold CV on 500k rows × 5 models × 25 Optuna trials = millions of fitting operations. Use `HalvingGridSearchCV`, early stopping, or subset CV to reduce compute.

5. **Serialisation:** A GBT model with 300 trees on 500k rows may exceed 100 MB. Consider `pickle` protocol 5 or model compression techniques.

</details>

---

**Q4.** How would you detect model degradation in production if you **cannot access ground-truth price labels** for new listings?

<details><summary>Show answer</summary>

Without labels you must rely on **proxy signals** and **input distribution monitoring**:

1. **Input drift (what we implemented):** Monitor Z-scores for each feature. If the input distribution shifts significantly (Z > 3), predictions are extrapolating beyond the training manifold — model reliability drops even if accuracy cannot be measured.

2. **Prediction distribution shift:** Monitor the distribution of model outputs (predicted prices). A sudden shift in the mean or variance of predictions often signals that the model is seeing out-of-distribution inputs.

3. **Delayed label collection:** If bookings are eventually observed, you can collect delayed ground truth (e.g., actual booked prices) weeks later. Even a 10% sample gives a periodic RMSE estimate.

4. **Business metric proxies:** If hosts who follow the model's recommendations have lower booking rates, that's a downstream signal of model degradation — even without seeing individual prediction errors.

5. **Confidence calibration:** Monitor the prediction confidence intervals. If CI widths grow systematically, the model's uncertainty is increasing — a sign it's encountering unfamiliar inputs.

</details>


## Week 12 Complete — Deliverables Checklist

### Project Milestone 2 Checklist

- [ ] **Feature engineering:** ≥ 5% R² improvement demonstrated over raw-feature Ridge baseline
- [ ] **5-algorithm comparison:** Ridge, RandomForest, GradientBoosting, SVR, XGBoost/ExtraTrees
- [ ] **Hyperparameter tuning:** Optuna (or GridSearchCV) applied to best algorithm
- [ ] **RMSE target:** Test RMSE < £30 (beats rule-based £42 baseline)
- [ ] **Production pipeline:** `ColumnTransformer` + sklearn `Pipeline` on raw DataFrame
- [ ] **Serialisation:** `joblib.dump` / `joblib.load` with assertion on predictions
- [ ] **REST endpoint:** `predict_endpoint()` function with CI output
- [ ] **Model Card:** Performance, intended use, limitations documented
- [ ] **Drift monitoring:** `check_drift()` function with Z-score alerts
- [ ] **Retrospective:** 4 questions answered

---

### Week 12 Summary

You have completed the following in Week 12:

| Topic | Key skill |
|-------|-----------|
| sklearn Pipeline | Prevent leakage, enable one-call CV |
| Custom Transformers | `BaseEstimator` + `TransformerMixin` |
| ColumnTransformer | Mixed-type data routing |
| Feature Engineering | Ratio, geographic, interaction, text features |
| Feature Selection | Mutual information + Lasso |
| Algorithm Comparison | 5-model benchmark |
| Hyperparameter Tuning | Optuna / GridSearchCV |
| Production Deployment | Serialisation + REST endpoint simulation |
| Responsible ML | Model Card + drift monitoring |

---

### Preview: Week 13 — Neural Networks

Next week we leave the sklearn world and enter **deep learning**:

- **Perceptrons and the biological neuron analogy** — activation functions as decision thresholds
- **Backpropagation from scratch** — chain rule visualised as computation graphs
- **PyTorch fundamentals** — tensors, autograd, `nn.Module`
- **Tabular networks** — beating GradientBoosting on the Airbnb dataset with a 3-layer MLP
- **Training diagnostics** — loss curves, gradient norms, learning rate schedules
- **Milestone 3** — Replace the production GBT with a PyTorch model and measure inference latency

**Prerequisite:** Make sure `torch` is installed: `pip install torch torchvision`
